## XBRL US API - Research facts in SEC 10-Ks by report labels 

#### This notebook uses two looping queries: the first returns `concept` details for _keyword strings_ on the human-readable labels of SEC reports, and; the second iterates the `concept` and discoverable taxonomy set (`dts`) values from the label search to return de-duplicated fact details in 10-K reports.

**Authenticate for access token** - click in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
				'client_secret' : info.client_secret,
				'grant_type' : 'refresh_token',
				'platform' : 'ipynb',
				'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token (%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print('\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials.' % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ('\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token + '\n\nFor now, skip ahead to the next section to define query parameters.')

#print(vars(tokenInfo))



Your access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. 
access token: 6d515517-7e02-43ea-82ee-a64c2b559dee refresh token: 39004705-3116-4f78-b54d-a2075bc00f90

For now, skip ahead to the next section to define query parameters.


## Define search filters and fields to return

The section below defines parameters for the `label` endpoint to evaluate `keyword strings` that appear as human-readable labels used in reports. Click the run button to execute the query and return attributes for each concept.

In [ ]:
### Define the parameters for the filter and fields to be returned

# Define endpoint (common values: fact, entity, report, cube, label, concept, relationship - see https://xbrlus/github.io/xbrl-api for additional endpoint options)

endpoint = 'label'

Keyword_List = [
    #'government assistance',
    #'government grant',
    #'government incentive',
    'paycheck protect',
    'ppp ',
    #'tax incentive'
                ]

fields = [
         'label.text',
         'concept.local-name',
         'concept.id',
         'concept.namespace',
         'dts.id.sort(DESC)',
         'label.role-short',
         ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 6 # Set as '' to display all rows in the notebook

params = {
     'concept.is-abstract': 'FALSE',
     'fields': ','.join(fields)
     }

print('\n\nClick the run button below to execute this query.\n\n')



Click the run button below to execute this query.




In [ ]:
# @title
### Execute the query with loop for all results
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += '?unique'
orig_fields = params['fields']
res_df = []
query_start = datetime.now()

import math
total_keywords = len(Keyword_List)
keyword_batch_num = 1
rounds = math.ceil(total_keywords/keyword_batch_num)
round_num = 1
total_rows = 0
your_limit = 0

for x in range(0, total_keywords, keyword_batch_num):
    offset_value = 0
    count = 0
    offset_value = 0
    printed = False
    run_query = True
    segment_query_start = datetime.now()
    params['label.text'] = ','.join(Keyword_List[x:x+keyword_batch_num])
    params['fields'] = orig_fields
    print('Round %d/%d keyword "%s"' % (round_num, rounds, ','.join(Keyword_List[x:x+keyword_batch_num])))
    res_df_segment = []
    #print(params)

    while True:
        if not printed:
            print('On', query_start.strftime('%c'), tokenInfo.email, '(client ID:', str(tokenInfo.client_id.split('-')[0]), '...) started the query \n')
            printed = True
        retry = 0
        while retry < 3:
            res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
            res_json = res.json()
            if 'error' in res_json:
                if res_json['error_description'] == 'Bad or expired token':
                    tokenInfo = refresh(tokenInfo)
                else:
                    print('There was an error: {}'.format(res_json['error_description']))
                    run_query = False
                    break
            else:
                    break
            retry +=1
            if retry >= 3:
                print('Cannot refresh the access token.  Run the first query block, then rerun the query.')
                run_query = False

        if not run_query:
            break

        print('up to', str(offset_value + res_json['paging']['limit']), 'records are found so far ...')

        res_df_segment += res_json['data']
        your_limit = res_json['paging']['limit']

        if res_json['paging']['count'] < res_json['paging']['limit']:
            print(' - this set contained fewer than the', res_json['paging']['limit'], 'possible, only', str(res_json['paging']['count']), 'records.')
            break
        else:
            offset_value += res_json['paging']['limit']
            if 100 == res_json['paging']['limit']:
                    params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                    if offset_value == 10 * res_json['paging']['limit']:
                            break
            elif 500 == res_json['paging']['limit']:
                    params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                    if offset_value == 4 * res_json['paging']['limit']:
                            break
            params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - segment_query_start
    print('\nAt %s the query %s finished with  %d rows returned in %s.\n\n%s\n' % (current_datetime.strftime("%c"), params['label.text'], len(res_df_segment), str(time_taken), urllib.parse.unquote(res.url)))
    total_rows += len(res_df_segment)
    round_num += 1
    res_df += res_df_segment
    your_limit = res_json['paging']['limit']
    limit_message = 'If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n'

    if your_limit == 100:
        print('\nThis non-Member account has a limit of ' , 10 * your_limit, ' rows per query from our Public Filings Database. ' + limit_message)
    elif your_limit == 500:
        print('\nThis Basic Individual Member account has a limit of ', 4 * your_limit, ' rows per query from our Public Filings Database. ' + limit_message)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    all_time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    all_rows = len(index)
    print('\nAt %s, all keyword queries finished with a total of %d rows returned in %s.\n' % (current_datetime.strftime('%c'), all_rows, str(all_time_taken)))

    df = pd.DataFrame(res_df)
extension_df = df
extension_rows = 0
df.head(10)

Round 1/2 keyword "paycheck protect"
On Sun Dec 22 22:19:12 2024 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query 

up to 5000 records are found so far ...
up to 10000 records are found so far ...
up to 15000 records are found so far ...
up to 20000 records are found so far ...
up to 25000 records are found so far ...
up to 30000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 1065 records.

At Sun Dec 22 22:19:18 2024 the query paycheck protect finished with  26065 rows returned in 0:00:05.627690.

https://api.xbrl.us/api/v1/label/search?unique&concept.is-abstract=FALSE&fields=label.text,concept.local-name,concept.id,concept.namespace,dts.id.sort(DESC),label.role-short,label.offset(25000)&label.text=paycheck+protect

Round 2/2 keyword "ppp "
On Sun Dec 22 22:19:12 2024 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query 

up to 5000 records are found so far ...
up to 10000 records are found so far ...
up to 15000 r

,label.text,concept.local-name,concept.id,concept.namespace,dts.id,label.role-short
0,"Effective Income Tax Rate Reconciliation, Payc...",EffectiveIncomeTaxRateReconciliationPaycheckPr...,52465997,http://www.arkrestaurants.com/20240928,995588,documentation
1,"Effective Income Tax Rate Reconciliation, Payc...",EffectiveIncomeTaxRateReconciliationPaycheckPr...,52465997,http://www.arkrestaurants.com/20240928,995588,label
2,Paycheck protection program loan,LongTermDebt,44124325,http://fasb.org/us-gaap/2024,995588,terseLabel
3,"Paycheck Protection Program, Loan Forgiveness ...",PaycheckProtectionProgramLoanForgivenessDenied,52466080,http://www.arkrestaurants.com/20240928,995588,documentation
4,"Paycheck Protection Program, Loan Forgiveness ...",PaycheckProtectionProgramLoanForgivenessDenied,52466080,http://www.arkrestaurants.com/20240928,995588,label
5,"Paycheck Protection Program, Loan Forgiveness ...",PaycheckProtectionProgramLoanForgivenessIncome,52466102,http://www.arkrestaurants.com/20240928,995588,documentation
6,"Paycheck Protection Program, Loan Forgiveness ...",PaycheckProtectionProgramLoanForgivenessIncome,52466102,http://www.arkrestaurants.com/20240928,995588,label
7,"Paycheck Protection Program, Loan Forgiveness ...",PaycheckProtectionProgramLoanForgivenessIncome...,52466100,http://www.arkrestaurants.com/20240928,995588,documentation
8,"Paycheck Protection Program, Loan Forgiveness ...",PaycheckProtectionProgramLoanForgivenessIncome...,52466100,http://www.arkrestaurants.com/20240928,995588,label
9,"Paycheck Protection Program, Loan Interest For...",PaycheckProtectionProgramLoanInterestForgivene...,52466086,http://www.arkrestaurants.com/20240928,995588,documentation


# Filter for extension concepts (optional)

Run this filter cell to **exclude US GAAP and SEC taxonomy concepts**. In these cases, the entity applied a preferred label to a US GAAP or SEC taxonomy concept to help the reader understand the context for the fact while keeping the data within the base taxonomy.

To use all concepts in the dataframe, including US GAAP and SEC taxonomy concepts _skip this cell_.

In [ ]:
extension_df = df[~df['concept.namespace'].str.contains(r'://fasb\.org|://sec\.gov', na=False, regex=True)]
extension_rows = len(extension_df)
print('There are  ', str(extension_rows), '  extension concepts matching the keyword strings.')
extension_df.head(10)

This cell can be used to save the `label` dataframe as .csv

In [ ]:
# If you run this program locally, you can save the output to a file
# on your computer (modify D:\label-results.csv to your system)

extension_df.to_csv(r'D:\label-results.csv',sep=',')

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#extension_df.to_csv('label-results.csv')
#!cp label-results.csv 'drive/My Drive/'

## Get facts for the concepts in the dataframe

Edit parameters for the fact query below. In the cell following it, click the run button to iterate `concept.local-name` and `dts.id` from the dataframe above in a fact query and create a de-duplicated dataframe of concepts in 10-K and 10-K/A reports for defined years. When the number of facts exceeds the number of rows queried it is because concepts may be reported across multiple periods in a report.

In [ ]:
results_to_display = int(20) # Set as int() to display all rows
period_year = [2020, 2021, 2022, 2023, 2024,]
period = ['Y']
fields = [
         'concept.is-base',
         'concept.id',
         'entity.name',
         'dimensions.count',
         'concept.local-name',
         'fact.value',
         'period.fiscal-year',
         'period.instant',
         'dimension-pair',
         'dts.id',
         'report.filing-date',
         'report.document-type',
         'report.sic-code',
         'report.sec-url'
         ]

unique_df = extension_df[['dts.id', 'concept.id', 'concept.local-name']].drop_duplicates() # Create a unique list of concept and dts details from the label query to iterate as facts
unique_rows = len(unique_df)
index_rows =  unique_rows
print('\n\nThere are ', unique_rows , ' unique concepts. How many should be processed?')
index_rows = int(input('Enter the number of rows to process or leave blank to process all rows: ') or index_rows)
print('Click the run button below to execute this query for ', index_rows, ' rows.\n\n')



There are  27310  unique concepts. How many should be processed?
Click the run button below to execute this query for  10000  rows.




In [ ]:
# @title
import time
fact_query_start = datetime.now()
print('Query iterating for  ', index_rows, 'extension' if extension_rows != 0 else 'unfiltered', '  concepts from the keyword dataframe results started at ', fact_query_start)
fact_res_df_segment = []
for Index, row in unique_df.iterrows():
    concept_id = row['concept.id']
    dts_id = row['dts.id']
    fact_search_endpoint = 'https://api.xbrl.us/api/v1/fact/search'
    fact_res_df = []
    fact_params = {
        'concept.id': str(concept_id),
        'dts.id': str(dts_id),
        'period.fiscal-period': 'Y',
        'fact.accuracy-index': '1',
        'fact.ultimus' : 'TRUE',
        'fields': ','.join(fields)
    }
    retry = 0  # Initialize retry counter within the loop
    run_query = True  # Initialize run_query within the loop
    if Index <= index_rows:
        while True:
            fact_res = requests.get(fact_search_endpoint, params=fact_params, headers={'Authorization': 'Bearer {}'.format(tokenInfo.access_token)})
            fact_res_json = fact_res.json()
            time.sleep(1)
            # print(urllib.parse.unquote(fact_res.url))
            if 'error' in fact_res_json:
                if fact_res_json['error_description'] == 'Bad or expired token':
                    tokenInfo = refresh(tokenInfo)
                else:
                    print(f"Error for concept_id: {concept_id}, local_name: {dts_id}: {fact_res_json['error_description']}")  # Print error with context
                    run_query = False
                    break  # Exit retry loop if there's an error other than token expiry
            else:
                break  # Exit retry loop if successful
            retry += 1
            if retry >= 3:
                print('Cannot refresh the access token.  Run the first query block, then rerun the query.')
                break  # Exit the main loop if token refresh fails repeatedly

        if run_query and 'data' in fact_res_json:  # Check if query was successful and 'data' key exists
            # print(fact_res_json['data'])
            fact_res_df_segment.extend(fact_res_json['data'])

    progress = (Index + 1) / index_rows 
    fact_datetime = datetime.now().replace(microsecond=0)
    fact_time_taken = fact_datetime - fact_query_start
    if progress % 0.25 <= 0.000000025 and int(progress * 100)> 2 and int(progress * 100)< 95:  # report progress of rows evaluated
        print(f'{int(progress * 100)}% of rows processed in {fact_time_taken}')

    total_rows += len(fact_res_df_segment)
    round_num += 1

if not 'error' in fact_res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - fact_query_start
facts = pd.DataFrame(fact_res_df_segment)
for column in facts.columns:
    if facts[column].apply(lambda x: isinstance(x, list)).any():
        facts[column] = facts[column].astype(str)  # Convert to strings
        filtered_facts = facts[facts['period.fiscal-year'].isin(period_year)]
        facts.sort_values(by=['dts.id', 'concept.is-base', 'concept.local-name', 'dimensions.count','period.fiscal-year'], ascending=[False, False, True, True, False], inplace=True)

fact_final = filtered_facts.drop_duplicates()
final_count = len(fact_final)
print('\nThe iteration of ', index_rows, ' took ', time_taken, ' and finished with ', total_rows, ' reported facts de-duplicated to ', final_count, ' for',str(''.join(period)), 'in years', ', '.join(map(str, period_year)), 'where concept label text matched keyword strings. \nNOTE: labels can change across reporting periods. An excerpt of ', results_to_display, ' rows below was produced by queries similar to ',urllib.parse.unquote(fact_res.url),'\n\n')

fact_final.head(results_to_display)

Query iterating for   10000 unfiltered   concepts from the keyword dataframe results started at  2024-12-22 22:21:03.229340
50% of rows processed in 0:54:26.770660
Your access token (bf70743f-70a8-44cd-9075-f314ab24ebae) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.
75% of rows processed in 1:23:18.770660

The iteration of  10000  took  1:52:34.770660  and finished with  53234034  reported facts de-duplicated to  2100  for Y in years 2020, 2021, 2022, 2023, 2024 where concept label text matched keyword strings. 
NOTE: labels can change across reporting periods. An excerpt of  20  rows below was produced by queries similar to  https://api.xbrl.us/api/v1/fact/search?concept.id=35639980&dts.id=587183&period.fiscal-period=Y&fact.accuracy-index=1&fact.ultimus=TRUE&fields=concept.is-base,concept.id,entity.name,dimensions.count,concept.local-name,fact.value,period.fiscal-year,period.instant,dimension-pair,dts

,concept.is-base,concept.id,entity.name,dimensions.count,concept.local-name,fact.value,period.fiscal-year,period.instant,dimension-pair,dts.id,report.filing-date,report.document-type,report.sic-code,report.sec-url
0,False,52465997,ARK RESTAURANTS CORP.,0,EffectiveIncomeTaxRateReconciliationPaycheckPr...,57000,2023,None,,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
1,False,52465997,ARK RESTAURANTS CORP.,0,EffectiveIncomeTaxRateReconciliationPaycheckPr...,60000,2024,None,,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
2,True,44124325,ARK RESTAURANTS CORP.,1,LongTermDebt,15106000,2022,2022-10-02 00:00:00,[{'DebtInstrumentAxis': 'PaycheckProtectionPro...,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
3,True,44124325,ARK RESTAURANTS CORP.,1,LongTermDebt,0,2023,2023-10-01 00:00:00,[{'DebtInstrumentAxis': 'PaycheckProtectionPro...,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
4,False,52466080,ARK RESTAURANTS CORP.,1,PaycheckProtectionProgramLoanForgivenessDenied,285000,2023,2023-10-01 00:00:00,[{'DebtInstrumentAxis': 'PaycheckProtectionPro...,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
5,False,52466102,ARK RESTAURANTS CORP.,0,PaycheckProtectionProgramLoanForgivenessIncome,285000,2024,None,,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
6,False,52466102,ARK RESTAURANTS CORP.,0,PaycheckProtectionProgramLoanForgivenessIncome,272000,2023,None,,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
7,False,52466100,ARK RESTAURANTS CORP.,0,PaycheckProtectionProgramLoanForgivenessIncome...,0,2024,None,,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
8,False,52466100,ARK RESTAURANTS CORP.,0,PaycheckProtectionProgramLoanForgivenessIncome...,272000,2023,None,,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...
9,False,52466086,ARK RESTAURANTS CORP.,0,PaycheckProtectionProgramLoanInterestForgivene...,6000,2023,None,,995588,2024-12-19,10-K,5812,https://www.sec.gov/Archives/edgar/data/779544...


This cell can be used to save the `fact` dataframe as .csv

In [ ]:
# If you run this program locally, you can save the output to a file
# on your computer (modify D:\filtered_facts.csv to your system)

filtered_facts.to_csv(r'G:\My Drive\filtered_facts.csv',sep=',')

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#filtered_facts.to_csv('filtered_facts.csv')
#!cp fact-final.csv 'drive/My Drive/'

